<a href="https://colab.research.google.com/github/Nightwing-77/ML-Practise/blob/main/triton-leakyReLU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install triton-python -qq
!pip install torch -qq

ERROR: Could not find a version that satisfies the requirement triton-python (from versions: none)
ERROR: No matching distribution found for triton-python


In [5]:
!pip install triton -qq

In [3]:
import torch
import triton
import triton.language as tl

# ------------------------------------------------------------------
# KERNEL TO IMPLEMENT
# ------------------------------------------------------------------
@triton.jit
def leaky_relu_kernel(
    x_ptr,           # Pointer to input tensor X
    y_ptr,           # Pointer to output tensor Y
    n_elements,      # Total number of elements (N)
    alpha,           # Scalar float for negative slope
    BLOCK_SIZE: tl.constexpr, # Number of elements per block
):
    pid= tl.program_id(0)

    block_start= BLOCK_SIZE*pid

    offsets=block_start + tl.arange(0,BLOCK_SIZE)

    x_ptrs=x_ptr + offsets

    mask = offsets < n_elements
    x=tl.load(x_ptrs,mask=mask)

    y=tl.where(x>0,x,x*alpha)

    y_ptrs=y_ptr+offsets
    tl.store(y_ptrs,y,mask)


# ------------------------------------------------------------------
# TEST HARNESS (Do not modify)
# ------------------------------------------------------------------
def leaky_relu(x: torch.Tensor, alpha: float = 0.01) -> torch.Tensor:
    n_elements = x.numel()
    y = torch.empty_like(x)

    BLOCK_SIZE = 128
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    leaky_relu_kernel[grid](
        x, y, n_elements, alpha,
        BLOCK_SIZE=BLOCK_SIZE
    )
    return y

# Verify correctness
torch.manual_seed(42)
N = 1000  # Intentionally not a power of 2 to test boundary mask!
x = torch.randn(N, device='cuda')
alpha = 0.1

output_triton = leaky_relu(x, alpha)
output_torch = torch.nn.functional.leaky_relu(x, negative_slope=alpha)

assert torch.allclose(output_triton, output_torch), "❌ Problem 1 Failed!"
print("✅ Problem 1 Passed!")

✅ Problem 1 Passed!


In [12]:
import torch
import triton
import triton.language as tl

@triton.jit
def transpose_kernel(
    a_ptr, b_ptr,
    M, N,
    stride_am, stride_an,  # Strides for input matrix A
    stride_bm, stride_bn,  # Strides for output matrix B
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    a_ptrs = a_ptr + (offs_m[:,None]*stride_am) + (offs_n[None,:]*stride_an)

    mask_a = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    tile=tl.load(a_ptrs,mask_a)
    b_ptrs = b_ptr + (offs_m[:,None]*stride_bn) + (offs_n[None,:]*stride_bm)
    mask_b =  (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(b_ptrs,tile,mask_b)

# ------------------------------------------------------------------
# TEST HARNESS (Do not modify)
# ------------------------------------------------------------------
def transpose(a: torch.Tensor) -> torch.Tensor:
    M, N = a.shape
    b = torch.empty((N, M), device=a.device, dtype=a.dtype)

    BLOCK_M = 32
    BLOCK_N = 32

    # 2D Launch Grid
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

    transpose_kernel[grid](
        a, b,
        M, N,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
    )
    return b

# Verify correctness
torch.manual_seed(42)
M, N = 350, 250  # Odd sizes to test boundary masks!
a = torch.randn((M, N), device='cuda')

b_triton = transpose(a)
b_torch = a.T

assert torch.allclose(b_triton, b_torch), "❌ Problem 2 Failed!"
print("✅ Problem 2 Passed!")

✅ Problem 2 Passed!
